In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
def get_range(output, target, n_step=-1):
    if n_step > 0:
        output = output[:, :n_step]
        target = target[:, 10:n_step]
    else:
        target = target[:, 10:]

    diff = target - output  # (b, t, x, y, c)

    max1 = np.maximum(np.max(target, axis=(-4, -3, -2)), np.max(output, axis=(-4, -3, -2)))
    min1 = np.minimum(np.min(target, axis=(-4, -3, -2)), np.min(output, axis=(-4, -3, -2)))

    max2 = np.max(diff, axis=(-4, -3, -2))
    min2 = np.min(diff, axis=(-4, -3, -2))

    return np.stack([max1, max2], axis=1), np.stack([min1, min2], axis=1)


def plot_ax_formal(
    ax, array, side_title=None, title=None, fontsize=15, cmap="RdBu_r", sym=False, ran_max=None, ran_min=None, pos=-1
):
    if sym:
        if ran_max is None:
            vmax = np.max(np.abs(array))
        else:
            vmax = max(ran_max, ran_min, -ran_min)
        im = ax.imshow(array, cmap=cmap, vmin=-vmax, vmax=vmax)
    else:
        if ran_max is None:
            im = ax.imshow(array, cmap=cmap)
        else:
            im = ax.imshow(array, cmap=cmap, vmin=ran_min, vmax=ran_max)

    if pos < 0 or pos == 2:
        cbar = plt.colorbar(im, ax=ax, fraction=0.046)
        cbar.ax.tick_params(labelsize=fontsize)

    if pos == 0:
        ax.annotate(side_title, (-0.1, 0.5), xycoords="axes fraction", rotation=90, va="center", fontsize=fontsize + 8)

    ax.tick_params(left=False, right=False, labelleft=False, labelbottom=False, bottom=False)
    if title is not None:
        ax.set_title(title, fontsize=fontsize)


def plot_2d(
    output: np.ndarray,
    data_all,
    plot_title="",
    input_len=10,
    output_plot_len=-1,
    dim=-1,
    save_path="",
    ran_max=None,
    ran_min=None,
):
    """
    output:     (output_len, x_num, x_num, data_dim)
    data_all:   (input_len + output_len, x_num, x_num, data_dim)
    """

    if dim < 0:
        dim = output.shape[-1]

    if output_plot_len < 0:
        output_len = output.shape[0]
        output_plot_len = output_len
    else:
        output_len = output_plot_len

    total_col = output_plot_len
    total_row = dim * 3
    fig, axs = plt.subplots(
        total_row, total_col, squeeze=False, figsize=(4 * total_col, 4 * total_row), layout="constrained"
    )

    for idx, output_step in enumerate(range(output_len - output_plot_len, output_len)):
        step = input_len + output_step
        if output_step + 1 == output_len:
            pos = 2
        elif output_step == 0:
            pos = 0
        else:
            pos = 1

        for j in range(dim):
            cur_target = data_all[step, ..., j]
            cur_output = output[output_step, ..., j]
            diff = cur_target - cur_output
            # diff = np.abs(cur_target - cur_output)

            if ran_max is None:
                plot_ax_formal(axs[j * 3 + 1, idx], cur_output, title="Prediction")
                plot_ax_formal(axs[j * 3, idx], cur_target, title=f"Target Step {output_step + 1}")
                plot_ax_formal(axs[j * 3 + 2, idx], diff, title="Difference", sym=True)
            else:
                plot_ax_formal(
                    axs[j * 3 + 1, idx], cur_output, "Prediction", ran_max=ran_max[0][j], ran_min=ran_min[0][j], pos=pos
                )
                plot_ax_formal(
                    axs[j * 3, idx],
                    cur_target,
                    "Target",
                    f"Step {output_step + 1}",
                    ran_max=ran_max[0][j],
                    ran_min=ran_min[0][j],
                    pos=pos,
                )
                plot_ax_formal(
                    axs[j * 3 + 2, idx],
                    diff,
                    "Difference",
                    sym=True,
                    ran_max=ran_max[1][j],
                    ran_min=ran_min[1][j],
                    pos=pos,
                )

    for ax in axs.flat:
        ax.label_outer()
        ax.tick_params(axis="both", labelsize=8)

    plt.suptitle(plot_title + "\n\n", fontsize=20)
    # plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()

In [ ]:
# exp2name = {
#     "bcat_1": "BCAT",
#     "vqbcat_2": "VQVAE+BCAT",
#     "bcat_2": "VAE+BCAT",
#     "bcat_5": "BCAT+reg toks",
#     "bcat_1_train": "BCAT",
#     "vqbcat_2_train": "VQVAE+BCAT",
#     "bcat_2_train": "VAE+BCAT",
#     "bcat_5_train": "BCAT+reg toks",
#     "bcat_muon_6": "BCAT Muon"
# }
exp2name = {
    "bcat_muon_all_1": "BCAT",
    "fno_test_1": "FNO",
    "unet_1": "UNet",
    "vit_1": "ViT",
    "vit_2": "ViT",
    "deeponet": "DeepONet",
    "mpp-b": "MPP-B",
    "mpp-l": "MPP-L",
    "dpot-m": "DPOT-M",
    "dpot-l": "DPOT-L",
    "prose_20": "PROSE-FD",
}


def plot_compare(
    target,
    all_data,
    plot_title="",
    output_len=-1,
    save_path="",
    i=0,
):
    dim = target.shape[-1]
    n_exp = len(all_data) + 1

    if output_len < 0:
        output_len = target.shape[1]

    total_col = output_len
    total_row = dim * n_exp
    fig, axs = plt.subplots(
        total_row, total_col, squeeze=False, figsize=(4.1 * total_col, 4 * total_row), layout="constrained"
    )

    for idx in range(output_len):
        if idx + 1 == output_len:
            pos = 2
        elif idx == 0:
            pos = 0
        else:
            pos = 1

        for j in range(dim):
            plot_ax_formal(axs[j * n_exp, idx], target[i, idx, ..., j], "Target", f"Step {idx + 1}", pos=pos)

            for k, (exp, d) in enumerate(all_data.items()):
                diff = d["diff"][i, idx, ..., j]
                title = "{} ({:.2%})".format(exp2name[exp], d["error"][i])
                plot_ax_formal(axs[j * n_exp + k + 1, idx], diff, title, sym=True, pos=pos)

    for ax in axs.flat:
        ax.label_outer()
        ax.tick_params(axis="both", labelsize=8)

    plt.suptitle(plot_title + "\n\n", fontsize=20)

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=400)
        plt.close(fig)
    else:
        plt.show()

Test

In [ ]:
expid = "1to1_auto_41"
exp = "cfdbench"
n_plot = 20
n_step = 6

path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
d = np.load(path)
output = d["output"][:, :n_step]
target = d["target"][:, : 10 + n_step]
errors = d["errors"]
ran_max, ran_min = get_range(output, target)

i = 0
# plot_2d(output[i], target[i])
plot_2d(output[i], target[i], ran_max=ran_max[i], ran_min=ran_min[i], plot_title=f"{exp} | Idx {i} | {errors[i]:.4f}")

Plot all for BCAT

In [ ]:
expid = "bcat_muon_all_1"

datasets = ["cfdbench", "com_ns", "incom_ns_arena_u", "incom_ns_arena", "incom_ns", "shallow_water"]
n_plot = 20
n_step = 6

for exp in datasets:
    path = f"/home/yuxuan/code/time/time-series/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, : 10 + n_step]
    errors = d["errors"]
    ran_max, ran_min = get_range(output, target)

    save_folder = f"/home/yuxuan/code/time/time-series/src/checkpoint/all_graphs/bcat_graph/{expid}/{exp}"
    os.makedirs(save_folder, exist_ok=True)

    for i in range(min(n_plot, errors.shape[0])):
        plot_2d(
            output[i],
            target[i],
            ran_max=ran_max[i],
            ran_min=ran_min[i],
            plot_title=f"{exp} | Idx {i} | {errors[i]:.4f}",
            save_path=f"{save_folder}/{exp}_{i}_{errors[i]:.3f}.png",
        )

Compare other models (BCAT)

In [ ]:
# exp = "incom_ns_arena"
exp = "com_ns"
expids = ["fno_test_1", "deeponet", "unet_1", "vit_2", "mpp-l", "dpot-l", "bcat_muon_all_1"]
# all_exps = ["bcat_1_train", "vqbcat_2_train", "bcat_2_train", "bcat_5_train"]
# all_exps = ["bcat_1", "bcat_2", "bcat_5", "bcat_muon_6"]

n_plot = 20
n_step = 6
all_data = {}

for expid in expids:
    folder = "time/time-series" if "muon" in expid else "prose/fluids"
    path = f"/home/yuxuan/code/{folder}/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, 10 : 10 + n_step]
    errors = d["errors"]
    all_data[expid] = {
        "exp": expid,
        "diff": target - output,
        "error": errors,
    }

save_folder = f"/home/yuxuan/code/time/time-series/src/checkpoint/all_graphs/bcat_graph/bcat_compare_update/{exp}"
os.makedirs(save_folder, exist_ok=True)

for i in range(min(n_plot, target.shape[0])):
    # for i in range(40, target.shape[0]):
    plot_compare(target, all_data, plot_title=f"{exp} | Idx {i}", save_path=f"{save_folder}/{exp}_{i}.png", i=i)

Compare Muon and AdamW

In [ ]:
exp2name = {
    "bcat_muon_all_1": "Muon",
    "1to1_auto_41": "AdamW",
}

exp = "incom_ns_arena"
# exp = "com_ns"
expids = ["1to1_auto_41", "bcat_muon_all_1"]

n_plot = 20
n_step = 6
all_data = {}
for expid in expids:
    if "muon" in expid:
        path = f"/data/yuxuan/code/checkpoints/time-series/save/{expid}/evals_all/outputs/{exp}.npz"
    else:
        path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"

    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, 10 : 10 + n_step]
    errors = d["errors"]

    all_data[expid] = {
        "exp": expid,
        "diff": target - output,
        "error": errors,
    }

# save_folder = f"/home/yuxuan/code/time/time-series/src/checkpoint/all_graphs/bcat_graph/compare_muon_adam/{exp}"
save_folder = f"/home/yuxuan/code/tmp_graph/bcat/{exp}"
os.makedirs(save_folder, exist_ok=True)

for i in range(min(n_plot, target.shape[0])):
    # for i in range(40, target.shape[0]):
    plot_compare(target, all_data, plot_title=f"{exp} | Idx {i}", save_path=f"{save_folder}/{exp}_{i}.png", i=i)

In [ ]:
def plot_compare2(
    target,
    all_data,
    plot_title="",
    output_len=-1,
    save_path="",
    i=0,
):
    dim = target.shape[-1]
    n_exp = len(all_data) + 1

    if output_len < 0:
        output_len = target.shape[1]

    total_col = output_len
    total_row = dim * n_exp
    fig, axs = plt.subplots(
        total_row, total_col, squeeze=False, figsize=(4.1 * total_col, 4 * total_row), layout="constrained"
    )

    for idx in range(output_len):
        if idx + 1 == output_len:
            pos = 2
        elif idx == 0:
            pos = 0
        else:
            pos = 1

        for j in range(dim):
            plot_ax_formal(axs[j * n_exp, idx], target[i, idx, ..., j], "Target", f"Step {idx + 1}", pos=pos)

            for k, (exp, d) in enumerate(all_data.items()):
                diff = d["diff"][i, idx, ..., j]
                title = "{} ({:.2%})".format(exp2name[exp], d["error"][i])
                plot_ax_formal(axs[j * n_exp + k + 1, idx], diff, title, sym=True, pos=pos, ran_max=0.9, ran_min=-0.9)

    for ax in axs.flat:
        ax.label_outer()
        ax.tick_params(axis="both", labelsize=8)

    plt.suptitle(plot_title + "\n\n", fontsize=20)

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=400)
        plt.close(fig)
    else:
        plt.show()


for i in [17]:
    plot_compare2(target, all_data, plot_title=f"{exp} | Idx {i}", i=i)

Double check the order is correct

In [ ]:
# datasets = ["cfdbench", "com_ns", "incom_ns_arena_u", "incom_ns_arena", "incom_ns", "shallow_water"]
datasets = ["com_ns", "incom_ns_arena_u", "incom_ns_arena", "shallow_water"]

exp = "incom_ns_arena_u"

bcat_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
# mpp_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
mpp_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/mpp-l/evals_all/outputs/{exp}.npz"
dpot_path = f"/home/yuxuan/code/prose/fluids/src/checkpoint/save/dpot-l/evals_all/outputs/{exp}.npz"

mpp = np.load(mpp_path)["target"]
bcat = np.load(bcat_path)["target"]
dpot = np.load(dpot_path)["target"][..., [1, 2, 0]]

diff1 = mpp - bcat
print(np.sum(np.abs(diff1)))

diff2 = dpot - bcat
print(np.sum(np.abs(diff2)))

Plot all for PROSE-FD

In [ ]:
expid = "prose_20"

datasets = ["cfdbench", "com_ns", "incom_ns_arena_u", "incom_ns_arena", "incom_ns", "shallow_water"]
n_plot = 20
n_step = 6

for exp in datasets:
    path = f"/home/yuxuan/code/time/time-series/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, : 10 + n_step]
    errors = d["errors"]
    ran_max, ran_min = get_range(output, target)

    save_folder = f"/home/yuxuan/code/time/time-series/src/checkpoint/all_graphs/prose_fd/{expid}/{exp}"
    os.makedirs(save_folder, exist_ok=True)

    for i in range(min(n_plot, errors.shape[0])):
        plot_2d(
            output[i],
            target[i],
            ran_max=ran_max[i],
            ran_min=ran_min[i],
            plot_title=f"{exp} | Idx {i} | {errors[i]:.4f}",
            save_path=f"{save_folder}/{exp}_{i}_{errors[i]:.3f}.png",
        )

Compare with other models (PROSE-FD)

In [ ]:
exp = "incom_ns_arena_u"
# exp = "com_ns"
expids = ["fno_test_1", "deeponet", "unet_1", "vit_2", "mpp-l", "dpot-m", "prose_20"]

n_plot = 20
n_step = 6
all_data = {}

for expid in expids:
    folder = "time/time-series" if "prose" in expid else "prose/fluids"
    path = f"/home/yuxuan/code/{folder}/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, 10 : 10 + n_step]
    errors = d["errors"]
    all_data[expid] = {
        "exp": expid,
        "diff": target - output,
        "error": errors,
    }

save_folder = f"/home/yuxuan/code/time/time-series/src/checkpoint/all_graphs/prose_fd/compare_models/{exp}"
os.makedirs(save_folder, exist_ok=True)

for i in range(min(n_plot, target.shape[0])):
    # for i in range(40, target.shape[0]):
    plot_compare(target, all_data, plot_title=f"{exp} | Idx {i}", save_path=f"{save_folder}/{exp}_{i}.png", i=i)

Compare PROSE-FD Adam vs Muon

In [ ]:
exp2name = {
    "prose_5": "AdamW",
    "prose_20": "Muon",
}

exp = "incom_ns_arena"
# exp = "com_ns"
expids = ["prose_5", "prose_20"]

n_plot = 20
n_step = 6
all_data = {}
for expid in expids:
    # path = f"/home/yuxuan/code/time/time-series/src/checkpoint/save/{expid}/evals_all/outputs/{exp}.npz"
    path = f"/data/yuxuan/code/checkpoints/time-series/save/{expid}/evals_all/outputs/{exp}.npz"
    d = np.load(path)
    output = d["output"][:, :n_step]
    target = d["target"][:, 10 : 10 + n_step]
    errors = d["errors"]

    all_data[expid] = {
        "exp": expid,
        "diff": target - output,
        "error": errors,
    }

# save_folder = f"/home/yuxuan/code/time/time-series/src/checkpoint/all_graphs/prose_fd/compare_muon_adam/{exp}"
save_folder = f"/home/yuxuan/code/tmp_graph/prose/{exp}"
os.makedirs(save_folder, exist_ok=True)

# for i in range(min(n_plot, target.shape[0])):
for i in range(40, target.shape[0]):
    plot_compare(target, all_data, plot_title=f"{exp} | Idx {i}", save_path=f"{save_folder}/{exp}_{i}.png", i=i)

In [ ]:
def plot_compare3(
    target,
    all_data,
    plot_title="",
    output_len=-1,
    save_path="",
    i=0,
):
    dim = target.shape[-1]
    n_exp = len(all_data) + 1

    if output_len < 0:
        output_len = target.shape[1]

    total_col = output_len
    total_row = dim * n_exp
    fig, axs = plt.subplots(
        total_row, total_col, squeeze=False, figsize=(4.1 * total_col, 4 * total_row), layout="constrained"
    )

    for idx in range(output_len):
        if idx + 1 == output_len:
            pos = 2
        elif idx == 0:
            pos = 0
        else:
            pos = 1

        for j in range(dim):
            plot_ax_formal(axs[j * n_exp, idx], target[i, idx, ..., j], "Target", f"Step {idx + 1}", pos=pos)

            for k, (exp, d) in enumerate(all_data.items()):
                diff = d["diff"][i, idx, ..., j]
                title = "{} ({:.2%})".format(exp2name[exp], d["error"][i])
                plot_ax_formal(axs[j * n_exp + k + 1, idx], diff, title, sym=True, pos=pos, ran_max=0.7, ran_min=-0.7)

    for ax in axs.flat:
        ax.label_outer()
        ax.tick_params(axis="both", labelsize=8)

    plt.suptitle(plot_title + "\n\n", fontsize=20)

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=400)
        plt.close(fig)
    else:
        plt.show()

In [ ]:
for i in [46]:
    plot_compare3(target, all_data, plot_title=f"{exp} | Idx {i}", i=i)